# LangChain Basics -- 6 ExperimentsRun each cell with **Shift + Enter**. The output appears directly below the cell.**Before you start:**1. Your virtual environment must be active2. `pip install -r requirements.txt` must have finished3. Your real API key must be in the `.env` file4. Top-right of this notebook -> **Select Kernel** -> **Python Environments** -> pick the `venv` oneIf a cell fails, read the **last** line of the red error text. That is the actualerror; everything above it is just the path the code took to get there.

---## Experiment 1 -- Your first AI callThis proves your key works and your environment is set up correctly.If this cell fails, nothing else will work, so fix it before moving on.

In [ ]:
import osfrom dotenv import load_dotenvfrom langchain_groq import ChatGroq# Reads the .env file and loads GROQ_API_KEY into the environment.load_dotenv()# Quick sanity check that the key was actually found:if not os.getenv("GROQ_API_KEY") or os.getenv("GROQ_API_KEY") == "PASTE_YOUR_GROQ_API_KEY_HERE":    print("STOP. Your API key is not set. Open the .env file and paste your real key.")else:    print("Key loaded. Length:", len(os.getenv("GROQ_API_KEY")), "characters")

In [ ]:
llm = ChatGroq(    model="llama-3.3-70b-versatile",    temperature=0.6)response = llm.invoke("What is the capital of India?")print(response.content)

**Expected output:** a sentence naming New Delhi.### What each piece does- **`load_dotenv()`** reads `.env` and makes `GROQ_API_KEY` available.  LangChain's Groq connector looks for that exact variable name automatically,  which is why you never pass the key in the code.- **`ChatGroq(...)`** creates the model object. Nothing is sent over the internet yet.- **`temperature=0.6`** controls creativity on a 0-to-1 scale.  At **0** you get the same safe answer every time -- use for factual work.  At **1** it takes risks, varies wildly, and is wrong more often.- **`.invoke(...)`** is the call that actually sends your text and waits for a reply.- **`.content`** pulls the plain text out. Without it you'd print a big object full of metadata.Try it: run the cell below to see what you get *without* `.content`.

In [ ]:
print(response)

---## Experiment 2 -- Prompt templatesHardcoding the question is useless in a real app -- the question comes from theuser at runtime. A prompt template is a fill-in-the-blank sentence with a slotfor the changing part.

In [ ]:
from langchain_core.prompts import PromptTemplatecapital_prompt = PromptTemplate(    input_variables=["country"],    template="Tell me the capital of {country}. Answer in one short sentence.")# Inspect the filled-in prompt WITHOUT sending it anywhere:print(capital_prompt.format(country="Japan"))

The `{country}` in curly braces is the slot. Whatever you pass as`country=` gets dropped into it.Nothing was sent to the AI here -- you only looked at the text that *would* be sent.This is a useful debugging habit: when a chain misbehaves, print the formattedprompt first and check the model is being asked what you think it's being asked.

---## Experiment 3 -- ChainsA chain connects components so the output of one flows into the next.Modern LangChain uses the pipe symbol `|`, borrowed from Unix.Read it left to right: *take the prompt, pipe it into the model, pipe that into the parser.*

In [ ]:
from langchain_core.output_parsers import StrOutputParserchain = capital_prompt | llm | StrOutputParser()print(chain.invoke({"country": "Japan"}))

**`StrOutputParser`** does one job: it converts the model's response objectinto a plain string, so you no longer write `.content` yourself. In a longer chainthis matters, because the next step usually expects a string, not an object.> **Note on old tutorials.** Most videos use `LLMChain(llm=llm, prompt=prompt)`.> That class still exists but is deprecated and prints a warning; it will be removed.> The pipe syntax above is called **LCEL** (LangChain Expression Language) and is> what interviewers expect to see. Learn the pipe, not `LLMChain`.

---## Experiment 4 -- Chaining two steps togetherNow the interesting part: take the answer from step one and feed it into step two.Ask for a country's capital, then ask for places to visit in *that capital* --without knowing in advance what the capital is.

In [ ]:
places_prompt = PromptTemplate(    input_variables=["capital"],    template="Suggest 3 famous places to visit in {capital}. Keep it brief.")# Step 1 produces a capital city name (a plain string).capital_chain = capital_prompt | llm | StrOutputParser()# Step 2 needs a dict with the key "capital", so convert the string into that shape.combined_chain = (    capital_chain    | (lambda text: {"capital": text})    | places_prompt    | llm    | StrOutputParser())print(combined_chain.invoke({"country": "India"}))

**Expected output:** a short list of places in New Delhi -- even though youonly ever typed "India".The small `lambda` in the middle is a converter. Step one hands over a plain string;step two expects a dictionary with a key called `capital`. The lambda reshapes thedata between them.You will do this constantly in real LangChain code. Most debugging time goes intomaking one step's output match the next step's expected input.

---## Experiment 5 -- Chat messages and rolesChat models understand three roles. This is how you control a chatbot'spersonality without retraining anything.| Role | What it is ||---|---|| **system** | Standing instructions for the whole conversation. The user never sees it. || **human** | What the user typed. || **ai** | What the model replied. Pass old ai messages back in to give it memory. |

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessagemessages = [    SystemMessage(content="You are a witty comedian. Answer everything with humour."),    HumanMessage(content="Explain what an API key is.")]response = llm.invoke(messages)print(response.content)

Now change the personality and run the cell below. **Same question,completely different answer.**That single line is the most powerful lever you have over a chatbot's behaviour --and it's exactly what the persona dropdown in `app.py` is swapping.

In [ ]:
messages = [    SystemMessage(content="You are a strict university professor. Be formal and precise."),    HumanMessage(content="Explain what an API key is.")]print(llm.invoke(messages).content)

---## Experiment 6 -- Custom output parsersSometimes you need structured data, not prose -- a Python list you can loop over,for example. A custom parser sits at the end of the chain and reshapes the text.

In [ ]:
from langchain_core.output_parsers import BaseOutputParserfrom langchain_core.prompts import ChatPromptTemplateclass CommaSeparatedListParser(BaseOutputParser):    def parse(self, text: str):        return [item.strip() for item in text.strip().split(",")]synonym_prompt = ChatPromptTemplate.from_messages([    ("system", "You are a helpful assistant. When the user gives a word, "               "reply with exactly 5 synonyms as a comma-separated list. "               "No numbering, no extra words, no full stop."),    ("human", "{word}")])synonym_chain = synonym_prompt | llm | CommaSeparatedListParser()result = synonym_chain.invoke({"word": "intelligent"})print(result)print(type(result))print("The third synonym is:", result[2])

**Expected output:** a real Python list such as`['smart', 'clever', 'bright', 'astute', 'sharp']`, then `<class 'list'>`,then the third item on its own.Compare that with the plain string you'd otherwise get. A string can't be indexedmeaningfully, looped over as items, or dropped into a dropdown menu. A list can.**This is the difference between a demo and something you can build a feature on.**

---## CheckpointIf all six experiments ran, you understand every LangChain concept the app needs:- models and temperature- prompt templates- chains with the pipe operator- message roles (system / human / ai)- output parsersEverything in `app.py` is assembly of these pieces plus Streamlit for the UI.**Next:** run `streamlit run app_simple.py` in your terminal to see the minimalversion, then `streamlit run app.py` for the full one.